## Import package

In [1]:
import numpy as np
import pandas as pd

Fundamentally, **data alignment is intrinsic**. The link between labels and data will not be broken unless done so explicity by you.

We'll give a brief intro to the data structures, then consider all of the broad categories of functionality and methods in seperate sections.

## DataFrame

<u>`DataFrame`</u> is a 2-dimensional labeled data structure with columns of potentially different types. You can think of it like a spreadsheet or SQL table, or a dict of Series objects. It is generally the most commonly used pandas object. Like Series, DataFrame accepts many different kinds of input:
* Dict of 1D ndarrays, lists, dicts, or <u>`Series`</u>
* 2-D numpy.ndarray
* <u>Structured or record</u> ndarray
* A <u>`Series`</u>
* Another <u>`DataFrame`</u>

Along with the data, you can optionally pass **index** (row labels) and **columns** (column labels) arguments. If you pass an index and / or columns, you are guaranteeing the index and / or columns of the resulting DataFrame. Thus, a dict of Series plus a specific index will discard all data not matching up to the passed index.

If axis labels are not passed, they will be constructed from the input data based on common sense rules.

### From dict of Series or dicts

The resulting **index** will be **union** of the indexes of the various Series. If there are any nested dicts, these will first be converted to Series. If no columns are passed, the columns will be the ordered list of dict keys.

In [2]:
d = {
    "one": pd.Series([1, 2, 3], index=["a", "b", "c"]),
    "two": pd.Series([1, 2, 3, 4], index=["a", "b", "c", "d"]),
}

In [3]:
df = pd.DataFrame(d)

In [4]:
df

,one,two
a,1.0,1
b,2.0,2
c,3.0,3
d,NaN,4


In [5]:
pd.DataFrame(d, index=["d", "b", "a"])

,one,two
d,NaN,4
b,2.0,2
a,1.0,1


In [6]:
pd.DataFrame(d, index=["d", "b", "a"], columns=["two", "three"])

,two,three
d,4,NaN
b,2,NaN
a,1,NaN


The row and column labels can be accessed respectively by accessing the **index** and **columns** attributes:

<table>
    <tr>
        <td>Note</td>
    </tr>
    <tr>
        <td>When a particular set of columns is passed along with a dict of data, the passed columns override the keys in the dict.</td>
    </tr>
</table>

In [7]:
df.index

Index(['a', 'b', 'c', 'd'], dtype='object')

In [8]:
df.columns

Index(['one', 'two'], dtype='object')

### From dict of ndarrays / lists

All ndarrays must share the same length. if an index is passed, it must also be the same length as the arrays. if no indes is passed, the result will be `range(n)`, where `m`m is the array length.

In [9]:
d = {"d": [1, 2, 3, 4], "two":[4, 3, 2, 1]}

In [10]:
pd.DataFrame(d)

,d,two
0,1,4
1,2,3
2,3,2
3,4,1


In [11]:
pd.DataFrame(d, index=["a", "b", "c", "d"])

,d,two
a,1,4
b,2,3
c,3,2
d,4,1


### From structured or record array

This case is handled identically to a dict of arrays.

In [12]:
data = np.zeros((2,), dtype=[("A", "i4"), ("B", "f4"), ("C", "a10")])

C:\Users\BAYU\AppData\Local\Temp\ipykernel_8584\1409807540.py:1: DeprecationWarning: Data type alias 'a' was deprecated in NumPy 2.0. Use the 'S' alias instead.
  data = np.zeros((2,), dtype=[("A", "i4"), ("B", "f4"), ("C", "a10")])


In [13]:
data[:] = [(1, 2.0, "Hello"), (2, 3.0, "World")]

In [14]:
pd.DataFrame(data)

,A,B,C
0,1,2.0,b'Hello'
1,2,3.0,b'World'


In [15]:
pd.DataFrame(data, index=["first", "second"])

,A,B,C
first,1,2.0,b'Hello'
second,2,3.0,b'World'


In [16]:
pd.DataFrame(data, columns=["C", "A", "B"])

,C,A,B
0,b'Hello',1,2.0
1,b'World',2,3.0


<table>
    <tr>
        <td>
            Note
        </td>
    </tr>
        <tr>
        <td>
            DataFrame is not intended to work eactly like 2-dimensional NumPy ndarray
        </td>
    </tr>
</table>

### From a list of dicts

In [17]:
data2 = [{"a": 1, "b": 2}, {"a": 5, "b": 10, "c": 20}]

In [18]:
pd.DataFrame(data2)

,a,b,c
0,1,2,NaN
1,5,10,20.0


In [19]:
pd.DataFrame(data2, index=["first", "second"])

,a,b,c
first,1,2,NaN
second,5,10,20.0


In [20]:
pd.DataFrame(data2, columns=["a", "b"])

,a,b
0,1,2
1,5,10


### From a dict of tuples

You can automatically create a MulteIndexed frame by passing a tuples dictionary

In [21]:
pd.DataFrame(
    {
        ("a", "b"): {("A", "B"): 1, ("A", "C"): 2},
        ("a", "a"): {("A", "C"): 3, ("A", "B"): 4},
        ("a", "c"): {("A", "B"): 5, ("A", "C"): 6},
        ("b", "a"): {("A", "C"): 7, ("A", "B"): 8},
        ("b", "b"): {("A", "D"): 9, ("A", "B"): 10},
    }
)

a              b      
       b    a    c    a     b
A B  1.0  4.0  5.0  8.0  10.0
  C  2.0  3.0  6.0  7.0   NaN
  D  NaN  NaN  NaN  NaN   9.0

### From a Series

The result will be a DataFame with the same index as the input Series, and with one column whose name is the original name of the Series (only if no column name provided).

In [22]:
ser = pd.Series(range(3), index=list("abc"), name="ser")

In [23]:
pd.DataFrame(ser)

,ser
a,0
b,1
c,2


### From a list of namedtuples

The field names of the first `namedtuple` in the list determine the columns of the <u>`DataFrame`</u>. The remaining namedtuples (or tuples) are simply unpacked and their values are fed into the rows of the <u>`DataFrame`</u>. if any of those tuples is shorter than the forst `namedtuple` then the later columns is the corresponding row are marked as missing valuse. If any are longer than the first `namedtuple`. a `ValueError` is raised.

In [24]:
from collections import namedtuple

In [25]:
Point = namedtuple("Pont", "x, y")

In [26]:
pd.DataFrame([Point(0, 0), Point(0, 3), (2, 3)])

,x,y
0,0,0
1,0,3
2,2,3


In [27]:
Point3D = namedtuple("Point3D", "x, y, z")

In [28]:
pd.DataFrame([Point3D(0, 0, 0), Point3D(0, 3, 5), Point(2, 3)])

,x,y,z
0,0,0,0.0
1,0,3,5.0
2,2,3,NaN


### From a list of dataclasses

Data Classes as introduced in <u>PEP557</u>, can be passed into the DataFrame constructor. Passing a list of dataclasses is equivalent to passing a list of dictionaries.

please be aware, that all values in the list should be dataclasses, mixing types int the list would result in a `TypeError`.

In [29]:
from dataclasses import make_dataclass

In [30]:
Point = make_dataclass("Point", [("x", int), ("y", int)])

In [31]:
pd.DataFrame([Point(0, 0), Point(0, 3), Point(2, 3)])

,x,y
0,0,0
1,0,3
2,2,3


**Missing data**

To construct a DataFrame with missing data, we use `np.nan` to represent missing values. Alternatively, you may pass a `numpy.MaskedArray` as the data argument to the DataFrame constructor, and its masked entries will be considered missing. See <u>Missing data</u> for more.

### Alternate constructors

**DataFrame.from_dict**

<u>`DataFrame.from_dict()`</u> takes a dict of dicts or a dict of array-like sequences and returns a DataFrame. It operates like the <u>`DataFrame`</u> constructor except fro the `orient` paramenter which is `'column'`by default, but which can be set to `'index'` in order to use the dict keys as row labels.

In [32]:
pd.DataFrame.from_dict(dict([("A",[1, 2, 3]), ("B", [4, 5, 6])]))

,A,B
0,1,4
1,2,5
2,3,6


if you pass `orient='index'`, the key will be the row labels, in this case, you can also pass the desired columns names:

In [33]:
pd.DataFrame.from_dict(
    dict([("A",[1, 2, 3]), ("B", [4, 5, 6])]),
    orient='index',
    columns=["one", "two", "three"]
)

,one,two,three
A,1,2,3
B,4,5,6


**DataFrame.from_records**

<u>`DataFrame.from_records()`</u> takes a list of tuples or an ndarray with structured dtype. It works analogously to the normal <u>`DataFrame`</u> constructor, except that the resulting DataFrame Index may be a specific field of the stryctured dtype.

In [34]:
data

array([(1, 2., b'Hello'), (2, 3., b'World')],
      dtype=[('A', '<i4'), ('B', '<f4'), ('C', 'S10')])

In [35]:
pd.DataFrame(data).set_index("C")

,A,B
C,,
b'Hello',1,2.0
b'World',2,3.0
